# Fact Theme Summary — Gold Layer

Aggregated fact table at the grain of one row per theme. Summarises set counts, part metrics, and timeline for each theme.

## What this notebook does

1. **Read** — Loads `fct_set_inventory`, `fct_set_minifigs`, and `dim_set` from gold.
2. **Transform** — Aggregates per-theme: total sets, average parts per set, total minifigs, year range, unique parts and colors.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/fct_theme_summary`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.fct_theme_summary`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, countDistinct, min as _min, max as _max,
    round as _round, sum as _sum, coalesce, lit,
)

spark = SparkSession.builder.getOrCreate()

GOLD_DIM_SET           = "lego_brickbase.gold.dim_set"
GOLD_FCT_SET_INVENTORY = "lego_brickbase.gold.fct_set_inventory"
GOLD_FCT_SET_MINIFIGS  = "lego_brickbase.gold.fct_set_minifigs"
DELTA_TABLE_PATH       = "/Volumes/lego_brickbase/gold/delta_volume/fct_theme_summary"
CATALOG_TABLE          = "lego_brickbase.gold.fct_theme_summary"

## Read and Transform

Aggregate set-level and inventory-level data up to the theme grain.

In [ ]:
df_dim_set = spark.table(GOLD_DIM_SET)
df_inv = spark.table(GOLD_FCT_SET_INVENTORY)
df_minifigs = spark.table(GOLD_FCT_SET_MINIFIGS)

# Set-level metrics per theme
df_set_agg = (
    df_dim_set
    .groupBy("theme_name", "root_theme_name")
    .agg(
        countDistinct("set_key").alias("total_sets"),
        _round(_sum("number_of_parts") / countDistinct("set_key"), 1).alias("avg_parts_per_set"),
        _sum("number_of_minifigs").alias("total_minifigs"),
        _min("year").alias("year_first_set"),
        _max("year").alias("year_latest_set"),
    )
)

df_set_agg = df_set_agg.withColumn(
    "active_span_years",
    col("year_latest_set") - col("year_first_set"),
)

# Inventory-level metrics per theme (unique parts, unique colors)
df_inv_agg = (
    df_inv
    .groupBy("theme_name")
    .agg(
        countDistinct("part_key").alias("unique_parts_used"),
        countDistinct("color_key").alias("unique_colors_used"),
    )
    .select(
        col("theme_name").alias("_inv_theme_name"),
        col("unique_parts_used"),
        col("unique_colors_used"),
    )
)

df_fct = (
    df_set_agg
    .join(df_inv_agg, df_set_agg["theme_name"] == df_inv_agg["_inv_theme_name"], "left")
    .select(
        df_set_agg["theme_name"],
        col("root_theme_name"),
        col("total_sets"),
        col("avg_parts_per_set"),
        coalesce(col("total_minifigs"), lit(0)).alias("total_minifigs"),
        col("year_first_set"),
        col("year_latest_set"),
        col("active_span_years"),
        coalesce(col("unique_parts_used"), lit(0)).alias("unique_parts_used"),
        coalesce(col("unique_colors_used"), lit(0)).alias("unique_colors_used"),
    )
)

df_fct.printSchema()
df_fct.display(10, truncate=False)

## Write to Delta Volume and Register Catalog Table

In [ ]:
(
    df_fct
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE}
    AS SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")
print(f"Catalog table ready: {CATALOG_TABLE}")